In [1]:
import pandas as pd
import numpy as np

In [2]:
patient_path = '../data/raw/data_clinical_patient.txt'
sample_path = '../data/raw/data_clinical_sample.txt'

patient_clinical = pd.read_csv(patient_path , sep='\t', comment='#')
sample_clinical = pd.read_csv(sample_path , sep='\t', comment='#')

dfs = [patient_clinical, sample_clinical]
for d in dfs:
    assert d['PATIENT_ID'].is_unique

patient_engineered = patient_clinical.copy().rename(str.lower, axis="columns")
sample_engineered = sample_clinical.copy().rename(str.lower, axis="columns")


patient_engineered['lymph_nodes_examined_positive'] = patient_engineered['lymph_nodes_examined_positive'].astype('Int64')


patient_engineered['cellularity'] = pd.Categorical(patient_engineered['cellularity'],
                                                    categories=['Low', 'Moderate','High'], ordered=True)
patient_engineered['cohort'] = patient_engineered['cohort'].astype('category')
patient_engineered['intclust'] = patient_engineered['intclust'].astype('category')


patient_engineered['claudin_subtype'] = patient_engineered['claudin_subtype'].replace({
    'NC' : np.nan
})


cols = ['chemotherapy', 'hormone_therapy', 'radio_therapy']
for col in cols:
    assert set(patient_engineered[col].dropna()) <= {'YES', 'NO'}
    patient_engineered[col] = patient_engineered[col].map({
    'YES' : True,
    'NO' : False
    }).astype('boolean')
    




patient_engineered = patient_engineered.drop(['sex', 'er_ihc', 'her2_snp6','histological_subtype'], axis=1)
sample_engineered = sample_engineered.drop(['sample_id', 'cancer_type','sample_type','cancer_type_detailed'], axis=1)

print(patient_engineered.columns)



sample_engineered['oncotree_code'] = (
    sample_engineered['oncotree_code']
    .where(
        sample_engineered['oncotree_code'].isin(['IDC', 'MDLC', 'ILC'])
        | sample_engineered['oncotree_code'].isna(),
        'Other'
    )
)




# for c in patient_engineered.select_dtypes(include=['category']):
#     print(patient_engineered[c].value_counts())

# print(patient_clinical.shape)
# print(sample_clinical.shape)


# patient_clinical.info()
# sample_clinical.info()


# display(patient_clinical.head())
# display(sample_clinical.head())

# data_dictionary = pd.read_csv(patient_path, sep='\t', header=None, nrows=4).T
# data_dictionary = data_dictionary.set_index(patient_clinical.columns)
# display(data_dictionary)

Index(['patient_id', 'lymph_nodes_examined_positive', 'npi', 'cellularity', 'chemotherapy', 'cohort', 'hormone_therapy', 'inferred_menopausal_state', 'intclust', 'age_at_diagnosis', 'os_months', 'os_status', 'claudin_subtype', 'threegene', 'vital_status', 'laterality', 'radio_therapy', 'breast_surgery', 'rfs_months', 'rfs_status'], dtype='str')


In [ ]:
merged_df = patient_engineered.merge(
    sample_engineered,
    how='left',
    on='patient_id',
    validate='one_to_one'
)

initial_size = len(merged_df)
merged_df = merged_df.dropna(subset=['os_months', 'vital_status'])
print(f'Removed row count: {initial_size - len(merged_df)}')

merged_df = merged_df.rename(columns={'os_months' : 'time'})
merged_df['event'] = merged_df['vital_status'] == 'Died of Disease'

Removed row count: 529


In [5]:
display(pd.crosstab(merged_df['os_status'], merged_df['vital_status'], dropna=False))

vital_status,Died of Disease,Died of Other Causes,Living
os_status,,,
0:LIVING,0,0,837
1:DECEASED,646,497,0


In [7]:
living = merged_df['os_status'] == '0:LIVING'
assert (merged_df.loc[living, 'vital_status'] == 'Living').all()
assert (merged_df.loc[~living, 'vital_status'] != 'Living').all()

n_patients = len(merged_df)
n_events = merged_df['event'].sum()
n_other_death = (merged_df['vital_status'] == 'Died of Other Causes').sum()
print(f'Patients: {n_patients}')
print(f'Died of breast cancer (events): {n_events}')
print(f'Censored: {n_patients - n_events}, of which died of other causes: {n_other_death}')

Patients: 1980
Died of breast cancer (events): 646
Censored: 1334, of which died of other causes: 497


In [8]:
display(merged_df['time'].describe())

count    1980.000000
mean      125.178737
std        76.075075
min         0.000000
25%        60.825000
50%       116.450000
75%       185.033333
max       355.200000
Name: time, dtype: float64

In [ ]:
zero_time = merged_df['time'] == 0.0
merged_df = merged_df.loc[~zero_time]
print(f'Number of 0 times: {zero_time.sum()}')


Number of 0 times: 0
1979


## Feature notes

**Outcome (never used as features):** `time`, `event`, `os_status`, `vital_status`, `rfs_months`, `rfs_status`

**Clinical features (model A):** age at diagnosis, tumour size, grade, lymph nodes positive, ER/PR/HER2 status, menopausal state, histology (`oncotree_code`), cellularity

**Molecular (gene-derived, model B only):** `claudin_subtype` (PAM50), `intclust`, `threegene`

**Open decisions:**
- NPI is calculated from tumour size, grade and lymph nodes, so use either NPI or its three parts, not both.
- Treatments (chemo, hormone, radio therapy) are chosen by doctors based on how risky the tumour looks, so they partly reflect the prognosis. Decide in the modelling notebook.
- `cohort` is the recruitment group, not a patient property, so it is not used as a feature.

In [ ]:
merged_df.to_parquet('../data/processed/clinical.parquet', index=False)